# Notebook 3: Utvärdering

## 3.1 Modelljämförelse

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("Data/modellresultat.csv")

mått = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)

for ax, kolumn in zip(axes, mått):
    ax.barh(df["Modell"], df[kolumn], color="black")
    ax.set_xlim(0.6, 1.0)
    ax.set_title(kolumn)
    ax.axvline(x=df[kolumn].max(), color="red", linestyle="--", linewidth=0.8)

plt.suptitle("Modelljämförelse — alla mått", fontsize=14)
plt.tight_layout()
plt.show()

## 3.2 ROC-kurvor & konfusionsmatris

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from matplotlib.colors import LinearSegmentedColormap

# Laddar data och tränar om de två bästa modellerna från notebook 2
df_raw = pd.read_csv("Data/clean_heart_disease.csv")
X = df_raw.drop(columns=["Hjärtsjukdom"])
y = df_raw["Hjärtsjukdom"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

pipe_pca_lr = Pipeline([("pca", PCA(n_components=5)), ("lr", LogisticRegression(max_iter=1000))])
pipe_pca_lr.fit(X_train_scaled, y_train)

# ROC-kurvor för de två bästa modellerna
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
fig.patch.set_facecolor("white")

for modell, namn in [(lr, "Logistisk Regression"), (pipe_pca_lr, "PCA + LR")]:
    fpr, tpr, _ = roc_curve(y_test, modell.predict_proba(X_test_scaled)[:, 1])
    auc = roc_auc_score(y_test, modell.predict_proba(X_test_scaled)[:, 1])
    ax.plot(fpr, tpr, linewidth=2, label=f"{namn}  (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Slumpmässig gissning")
ax.set_xlabel("Falskt larm (FPR)", fontsize=12)
ax.set_ylabel("Träffsäkerhet (Recall / TPR)", fontsize=12)
ax.set_title("ROC-kurvor — de två bästa modellerna", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Konfusionsmatris för bästa modellen (LR)
rg_cmap = LinearSegmentedColormap.from_list("rg", ["#c0392b", "#27ae60"])
y_pred_lr = lr.predict(X_test_scaled)
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr), display_labels=["Frisk", "Sjuk"])
fig, ax = plt.subplots(figsize=(5, 4), dpi=200)
disp.plot(ax=ax, colorbar=True, cmap=rg_cmap)
for text in disp.text_.ravel():
    text.set_color("white")
ax.set_xlabel("Förutsagt", fontsize=12)
ax.set_ylabel("Faktiskt", fontsize=12)
ax.set_title("Logistisk Regression — Konfusionsmatris", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 3.3 Kan modellen förbättras?

In [ ]:
from sklearn.model_selection import cross_val_score

# Korsvalidering delar träningsdatan i 5 delar och roterar vilken som är valideringsdata
# Ger ett mer pålitligt resultat än ett enda train/test-split
cv_recall = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring="recall")
cv_auc    = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring="roc_auc")

print("Logistisk Regression — korsvalidering (5-fold):")
print(f"  Recall:  {cv_recall.mean():.3f} ± {cv_recall.std():.3f}")
print(f"  ROC-AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

# Testar olika C-värden (regulariseringsstyrka)
# Lågt C = enklare modell, högt C = mer anpassad till träningsdatan
C_värden = [0.001, 0.01, 0.1, 1, 10, 100]
recall_scores = []
auc_scores    = []

for C in C_värden:
    m = LogisticRegression(C=C, max_iter=1000, random_state=42)
    recall_scores.append(cross_val_score(m, X_train_scaled, y_train, cv=5, scoring="recall").mean())
    auc_scores.append(cross_val_score(m, X_train_scaled, y_train, cv=5, scoring="roc_auc").mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
fig.patch.set_facecolor("white")

for ax, scores, titel in [(ax1, recall_scores, "Recall"), (ax2, auc_scores, "ROC-AUC")]:
    ax.plot(range(len(C_värden)), scores, marker="o", color="black")
    ax.set_xticks(range(len(C_värden)))
    ax.set_xticklabels(C_värden)
    ax.set_xlabel("C (regulariseringsstyrka)", fontsize=11)
    ax.set_ylabel(titel, fontsize=11)
    ax.set_title(f"{titel} vs C", fontsize=12)

plt.suptitle("Logistisk Regression — effekt av regularisering", fontsize=13)
plt.tight_layout()
plt.show()

bästa_C = C_värden[np.argmax(recall_scores)]
print(f"\nBästa C baserat på Recall: {bästa_C}")

## 3.4 Slutsatser

Projektet testade sex varianter av två modeller — Logistisk Regression och KNN — på ett dataset med 918 patienter och 20 variabler.

**Bästa modell: Logistisk Regression (utan komprimering)**

I ett medicinskt sammanhang är Recall det viktigaste måttet — det är värre att missa en sjuk patient än att få ett falskt larm. LR presterade bäst på just det:

| Modell | Recall | ROC-AUC |
|---|---|---|
| Logistisk Regression | **0.912** | **0.933** |
| PCA + LR | 0.902 | 0.931 |
| KNN | 0.902 | 0.922 |

**Vad vi lärde oss:**
- Enklare modeller slår ofta mer komplexa — LR utan komprimering vann på det viktigaste måttet
- Komprimering hjälper inte alltid — UMAP försämrade LR kraftigt trots att det är en mer avancerad metod
- Modell och data måste passa ihop — PCA bevarade de linjära mönster som LR behöver, UMAP förstörde dem

**Begränsningar:**
- Datasetet är relativt litet (918 patienter) — resultaten kan variera på annan data
- Modellen är inte testad på extern data utanför detta dataset
- Klinisk användning skulle kräva betydligt mer validering och godkännande